# Output 7 — Risk rating summary

Excel analogue: **Output 7 - Risk rating summary**.
Mechanical ratings come from Chart Data (baseline + standard B-tests).
Finals default to mechanical until judgement is applied.

See `docs/10-risk-rating.qmd`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.load import (
    load_ci_summary,
    load_core,
    load_input6_standard,
    load_input7_residual_params,
)

from lic_dsf.rating import (
    ChartDataRegistry,
    RiskRatingSummary,
    compute_mechanical_ratings,
    moderate_panel,
    risk_summary_panel,
)
from lic_dsf.stress import (
    run_b1_gdp_public,
    run_standard_external_stress,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 16)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK

PosixPath('/home/sravan/excel-grapher/py-lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

In [2]:
macro, external, ext_base, pub_base = load_core(WORKBOOK)
ci = load_ci_summary(WORKBOOK)
input6 = load_input6_standard(WORKBOOK)
residual = load_input7_residual_params(WORKBOOK)
external_stress = run_standard_external_stress(macro, external, input6, residual)
public_b1 = run_b1_gdp_public(macro, external, input6, residual)

first_proj = int(macro.inputs.first_projection_year)
proj_years = list(range(first_proj, first_proj + 11))
ci.country, ci.dcc.value, round(ci.ci_score, 4), proj_years[0], proj_years[-1]

('Ghana', 'Medium', 2.7399, 2024, 2034)

## Output 7 panel

`risk_summary_panel` is the Output 7-shaped table. Moderate granularity is
`n.a.` unless the mechanical external rating is Moderate.

In [3]:
registry = ChartDataRegistry()
_EXTERNAL = (
    ("pv_debt_to_gdp", "pv_ppg_external_to_gdp"),
    ("pv_debt_to_exports", "pv_ppg_external_to_exports"),
    ("debt_service_to_exports", "ppg_debt_service_to_exports"),
    ("debt_service_to_revenue", "ppg_debt_service_to_revenue"),
)
for indicator, method in _EXTERNAL:
    registry.register_series(
        indicator,
        "baseline",
        getattr(ext_base, method)().reindex(proj_years),
        is_baseline=True,
    )
    for sid, book in external_stress.items():
        registry.register_series(
            indicator,
            sid,
            getattr(book, method)().reindex(proj_years),
            is_shock=True,
        )
registry.register_series(
    "public_pv_debt_to_gdp",
    "baseline",
    pub_base.pv_public_debt_to_gdp().reindex(proj_years),
    is_baseline=True,
)
registry.register_series(
    "public_pv_debt_to_gdp",
    "B1_GDP",
    public_b1.pv_public_debt_to_gdp().reindex(proj_years),
    is_shock=True,
)

mechanical = compute_mechanical_ratings(registry, ci.thresholds, years=proj_years)
out_5_1 = moderate_panel(
    mechanical_external=mechanical.external,
    baseline_pv_gdp=ext_base.pv_ppg_external_to_gdp(),
    threshold_pv_gdp=ci.thresholds.pv_debt_to_gdp,
    rating_years=proj_years,
)
summary = RiskRatingSummary(
    mechanical=mechanical,
    thresholds=ci.thresholds,
    dcc=ci.dcc,
    ci_score=ci.ci_score,
    moderate_granularity=str(out_5_1.loc["Space to absorb shock", "Output 5-1"]),
)
out_7 = risk_summary_panel(summary)
out_7

,Output 7
Mechanical external,High
Final external,High
Mechanical fiscal,High
Mechanical overall,High
Final overall,High
Judgement applied,No
Debt carrying capacity,Medium
CI score,2.7399
Threshold PV/GDP,40.0000
Threshold PV/exports,180.0000
